In [2]:

import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [6]:
import json
import pandas as pd
import argparse
from pathlib import Path

def extract_flagged_entities(json_file):
    """Extract flagged entities with context from supervisor output JSON file."""
    
    # Load the JSON file
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Get the flagged entities from the summary
    flagged_entities = data.get('summary', {}).get('flagged_entities', [])
    print(f"Found {len(flagged_entities)} flagged entities")
    
    # Prepare detailed data for review
    detailed_entities = []
    
    # Process each flagged entity
    for entity in flagged_entities:
        entity_info = {
            'entity': entity.get('entity', ''),
            'document_id': entity.get('document_id', ''),
            'orpha_code': entity.get('orpha_code', ''),
            'category': entity.get('category', ''),
            'explanation': entity.get('explanation', '')
        }
        
        # Get detailed information from the results section
        category = entity.get('category', '')
        doc_id = entity.get('document_id', '')
        
        if doc_id and category and category in data.get('results', {}):
            # Find the detailed entity data
            for result in data['results'][category]:
                if (result.get('document_id') == doc_id and 
                    result.get('entity') == entity_info['entity']):
                    # Add detailed info
                    entity_info['context'] = result.get('context', '')
                    entity_info['is_rare_disease'] = result.get('is_rare_disease', False)
                    
                    # Add top candidate
                    candidates = result.get('orpha_candidates', [])
                    if candidates:
                        top_candidate = candidates[0]
                        entity_info['top_candidate_name'] = top_candidate.get('name', '')
                        entity_info['top_candidate_id'] = top_candidate.get('id', '')
                        entity_info['top_candidate_similarity'] = top_candidate.get('similarity', 0.0)
                    
                    break
        
        detailed_entities.append(entity_info)
    
    return detailed_entities

def main():
    # parser = argparse.ArgumentParser(description="Extract flagged entities for review")
    # parser.add_argument("json_file", help="Path to supervisor output JSON file")
    # parser.add_argument("--output", help="Output CSV file for review (optional)")
    # parser.add_argument("--category", choices=["false_positives", "false_negatives", "true_positives"],
    #                   help="Filter by category (optional)")
    
    # args = parser.parse_args()
    
    # Extract entities
    json_file = "data/results/supervisor/multistage_no_min.json"
    category = "false_positives"
    entities = extract_flagged_entities(json_file)
    print(entities[0])
    # Filter by category if requested
    if category:
        entities = [e for e in entities if e.get('category') == category]
        print(f"Filtered to {len(entities)} {category}")
    
    # Create dataframe
    df = pd.DataFrame(entities)
    
    
if __name__ == "__main__":
    main()

Found 16 flagged entities
{'entity': 'HIT', 'document_id': '10406', 'orpha_code': '3325', 'category': 'false_negatives', 'explanation': "The entity 'HIT' in the given context refers to 'Heparin-Induced Thrombocytopenia', which is a recognized condition. However, the context mentions that 'A HIT-Ab was sent which was negative', indicating that the condition was ruled out. Therefore, it is negated in the context.  [FLAGGED: Entity determined not to be a rare disease despite being categorized as false negative]", 'context': 'ent had transient thrombocytopenia\nwhile in the ICU that resolved prior to transfer to the floor.\nA HIT-Ab was sent which was negative.  He had no evidence of\npetechiae or easy bruising.\n\n4. Skin lesions', 'is_rare_disease': False, 'top_candidate_name': 'zinc finger hit-type containing 3', 'top_candidate_id': 'ORPHA:487198', 'top_candidate_similarity': 0.6394930288648274}
Filtered to 6 false_positives
